# Can Muse Spark coach the photographer? (capped pilot)

**Meta model:** Muse Spark 1.3 (`muse-spark-1.3`, Standard mode) via the Meta Model API.
**Hard caps: 20 requests and $0.25.** The run stops at whichever comes first and prints the real cost after every call.

20 DeepDRiD validation photos: 5 good, 5 blurry only, 5 misaimed only, 5 obstructed only (by DeepDRiD's own scores).
Muse Spark sees each photo and must say whether to retake, what the problem is, and one instruction for the technician.
Needs `MODEL_API_KEY` in Colab Secrets. Any runtime (no GPU).

In [ ]:
import os, glob, json, base64, io, time
from google.colab import drive, userdata
drive.mount('/content/drive')
D = '/content/drive/MyDrive/ebl_fundus'
os.makedirs(f'{D}/quality', exist_ok=True)
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/deepdrdoc/DeepDRiD.git /content/deepdrid 2>/dev/null; true
!cd /content/deepdrid && git sparse-checkout set regular_fundus_images/regular-fundus-validation && git checkout -q
!pip install -q openai
os.environ['MODEL_API_KEY'] = userdata.get('MODEL_API_KEY')

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
from openai import OpenAI

V = '/content/deepdrid/regular_fundus_images/regular-fundus-validation'
dd = pd.read_csv(f'{V}/regular-fundus-validation.csv')
c, f, a = dd['Clarity'] <= 6, dd['Field definition'] <= 6, dd['Artifact'] >= 6
groups = {'good': (dd['Overall quality'] == 1) & ~c & ~f & ~a, 'focus': c & ~f & ~a, 'aim': f & ~c & ~a, 'obstruction': a & ~c & ~f}
pick = pd.concat([dd[m].sample(min(5, m.sum()), random_state=0).assign(truth=g) for g, m in groups.items()])
print(pick.truth.value_counts().to_dict())

PROMPT = """You are helping a technician who takes retinal (fundus) photos for diabetic eye screening.
Judge only PHOTO QUALITY, not disease. Problems:
- focus: blurry or hazy, small vessels not sharp
- aim: optic disc and macula not both near the centre of the field
- obstruction: eyelid or eyelash shadow, glare, reflections or dust covering part of the retina
- exposure: too dark or washed out
Reply with JSON only: {"retake": true or false, "problems": [zero or more of "focus", "aim", "obstruction", "exposure"],
"instruction": "one short sentence telling the technician what to do"}"""

client = OpenAI(base_url='https://api.meta.ai/v1', api_key=os.environ['MODEL_API_KEY'])
PRICE_IN, PRICE_OUT = 1.25e-6, 4.25e-6          # muse-spark-1.3, Standard mode, $ per token
MAX_REQUESTS, MAX_USD = 20, 0.25

def as_data_url(path):
    img = Image.open(path).convert('RGB'); img.thumbnail((1024, 1024))
    buf = io.BytesIO(); img.save(buf, 'JPEG', quality=90)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

In [ ]:
spent, worst, rows = 0.0, 0.0, []
for n, r in enumerate(pick.itertuples(), 1):
    if n > MAX_REQUESTS or spent + worst > MAX_USD:     # stop if the next call could cross the cap
        print(f'stopping before request {n}: ${spent:.3f} spent, next call could cost up to ${worst:.4f}'); break
    resp = client.chat.completions.create(
        model='muse-spark-1.3', extra_body={'reasoning_effort': 'low'},
        messages=[{'role': 'user', 'content': [{'type': 'text', 'text': PROMPT},
                  {'type': 'image_url', 'image_url': {'url': as_data_url(f'{V}/Images/{r.patient_id}/{r.image_id}.jpg')}}]}])
    cost = resp.usage.prompt_tokens * PRICE_IN + resp.usage.completion_tokens * PRICE_OUT
    spent, worst = spent + cost, max(worst, cost)
    text = resp.choices[0].message.content or ''
    try:
        ans = json.loads(text[text.index('{'):text.rindex('}') + 1])
    except ValueError:
        ans = {}
    rows.append(dict(image_id=r.image_id, truth=r.truth, retake=ans.get('retake'), problems=ans.get('problems'),
                     instruction=ans.get('instruction'), tokens_in=resp.usage.prompt_tokens,
                     tokens_out=resp.usage.completion_tokens, cost=cost))
    print(f"{n:2d} {r.truth:11s} -> retake={ans.get('retake')} problems={ans.get('problems')}  "
          f"(${cost:.4f}, total ${spent:.3f})")
res = pd.DataFrame(rows)
res.to_csv(f'{D}/quality/muse_coach_pilot.csv', index=False)

In [ ]:
res['retake_right'] = res.retake == (res.truth != 'good')
res['problem_named'] = [np.nan if t == 'good' else float(isinstance(p, list) and t in p) for t, p in zip(res.truth, res.problems)]
print(res.groupby('truth')[['retake_right', 'problem_named']].mean().round(2).to_string())
print(f"\n{len(res)} requests, total ${res.cost.sum():.3f} (${res.cost.mean():.4f} per photo)\n")
for t in ['focus', 'aim', 'obstruction']:
    ex = res[res.truth == t].instruction.dropna()
    if len(ex): print(f'{t:11s} example instruction: {ex.iloc[0]}')